# Main Figure 2

Plot BEF slopes across degree and interaction strength.

Input: generated/simulations.csv and generated/run_config.json. Run 00_generate_simulations.ipynb first. Set DATA_DIR for a different run.

Fit biomass against initial richness for each parameter pair.

## Settings

Change DATA_DIR and SAVE in the next cell.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

DATA_DIR = Path('generated')
SAVE = True
FIGURE_DIR = Path('figures')
FOCUS_DEGREE = 1.0
FOCUS_STRENGTH = 0.5
simulations = pd.read_csv(DATA_DIR / 'simulations.csv')
config = json.loads((DATA_DIR / 'run_config.json').read_text(encoding='utf-8'))
FUNCTION_COL = 'biomass_for_bef'
if config['profile'] != 'final_10000':
    raise ValueError('Use data from the final_10000 simulation notebook')

def slope_table(df, x='N', y=None, groups=('degree','strength')):
    y = FUNCTION_COL if y is None else y
    rows = []
    for keys, part in df.groupby(list(groups)):
        if part[x].nunique() < 2:
            continue
        fit = linregress(part[x], part[y])
        rows.append(dict(zip(groups, keys if isinstance(keys, tuple) else (keys,))) |
                    dict(slope=fit.slope, intercept=fit.intercept, stderr=fit.stderr,
                         n=len(part), richness_levels=part[x].nunique()))
    return pd.DataFrame(rows)

def set_strength_axis(ax, values):
    lo, hi = float(np.min(values)), float(np.max(values))
    ticks = sorted(set([lo, hi] + [float(x) for x in np.linspace(.2, 1, 5) if lo < x < hi]))
    ax.set_xlim(lo, hi)
    ax.set_xticks(ticks, [f'{x:g}' for x in ticks])

def finish(fig, stem):
    fig.tight_layout()
    if SAVE:
        FIGURE_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURE_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')
        fig.savefig(FIGURE_DIR / f'{stem}.pdf', bbox_inches='tight')
    plt.show()


## Figure

In [ ]:
slopes = slope_table(simulations)
print(slopes.head(), '\nSlope table:', slopes.shape)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
target_strength = min(slopes.strength.unique(), key=lambda x: abs(x - FOCUS_STRENGTH))
target_degree = min(slopes.degree.unique(), key=lambda x: abs(x - FOCUS_DEGREE))
a = slopes[slopes.strength == target_strength].sort_values('degree')
b = slopes[slopes.degree == target_degree].sort_values('strength')
for ax, cut, xcol, fixed_col, fixed_value, label in [
    (axes[0], a, 'degree', 'strength', target_strength, 'a'),
    (axes[1], b, 'strength', 'degree', target_degree, 'b')]:
    colors = plt.cm.magma_r(np.linspace(.15, .75, len(cut)))
    ax.scatter(cut[xcol], cut.slope, c=colors, s=45, zorder=3)
    ax.plot(cut[xcol], cut.slope, color='#aa4477', lw=1, alpha=.65)
    ax.set(xlabel='Connectance, mean degree' if xcol == 'degree' else 'Interaction strength, max(omega)',
           ylabel='BEF slope')
    ax.text(-.1, 1.02, label, transform=ax.transAxes, fontsize=17, fontweight='bold')
    inset = ax.inset_axes([.48, .52, .47, .43])
    for color, value in zip(colors, cut[xcol]):
        rows = simulations[np.isclose(simulations[fixed_col], fixed_value) &
                           np.isclose(simulations[xcol], value)]
        means = rows.groupby('N')[FUNCTION_COL].mean()
        inset.plot(means.index, means.values, color=color, lw=1.3, alpha=.9)
    inset.set(xlabel='Biodiversity (N)', ylabel='Biomass')
    inset.tick_params(labelsize=7)
    inset.xaxis.label.set_size(8); inset.yaxis.label.set_size(8)
fig.tight_layout()
finish(fig, 'main_figure_2')